# Manglish LoRA — Colab fine-tune recipe

Runs the repo's harness end-to-end on a free **T4 GPU**: validate data →
LoRA fine-tune → holdout evaluation with visible generations.

**Setup:** upload the repo as a zip (or git clone), `Runtime → Change runtime
type → GPU (T4)`, then run top to bottom. See `README.md` for the honest
status: the sample corpus is plumbing-grade; a real corpus is the prerequisite
for meaningful output.

In [ ]:
# 1 — get the repo into the session (clone, or upload a zip and unzip)
# !git clone https://github.com/flogduse/manglish-lora.git
# %cd manglish-lora
# --- or upload manglish-lora.zip and: ---
# !unzip -q manglish-lora.zip
# %cd manglish-lora

In [ ]:
# 2 — GPU stack (torch ships with Colab; add the fine-tune extras)
!pip install -q -r requirements-ml.txt

In [ ]:
# 3 — validate + split FIRST: bad data must fail here, before GPU minutes burn
!python scripts/prepare_data.py

In [ ]:
# 4 — sanity check without spending real time: the sample is tiny, so this
# finishes in minutes and proves the whole path works.
!python scripts/train_lora.py \
    --base TinyLlama/TinyLlama-1.1B-Chat-v1.0 \
    --data data/train.jsonl \
    --epochs 3 --r 16 --alpha 32 \
    --out runs/manglish-lora

In [ ]:
# 5 — holdout perplexity + visible sample generations
!python scripts/evaluate.py --data data/eval.jsonl --model runs/manglish-lora

In [ ]:
# 6 — keep the adapter only (LoRA is a few MB, not the 4-bit base)
!zip -q -r manglish-lora-adapter.zip runs/manglish-lora
from google.colab import files
files.download("manglish-lora-adapter.zip")

## Next iterations

- Swap the base: `--base microsoft/phi-2` or `google/gemma-2b` for quality
  comparisons at the same LoRA budget.
- Grow the corpus first — the README lists licence-checked sources.
- Once results hold up, publish the adapter on Hugging Face with the sample
  corpus and this notebook as provenance.